### AI Financial Analyst

#### Member 2

### Notebook 3 - Narrative Section Extraction

---

#### 1. Objective

The objective of this notebook is to extract the narrative sections required
for the Retrieval-Augmented Generation (RAG) pipeline.

Using the company mapping prepared by Member 1, each selected company-year
is matched with its corresponding filing in the historical EDGAR dataset.

The following narrative sections are extracted:

- Section 1 - Business
- Section 1A - Risk Factors
- Section 7 - Management Discussion and Analysis (MD&A)

The extracted narrative data will be used in the subsequent text
preprocessing and embedding stages.

---

#### CRISP-DM Phase

Data Preparation

#### Step 1 - Importing libraries

In [1]:
# Importing Libraries

import json
from pathlib import Path

import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


#### Step 2 - Loading Member 1 outputs

In [2]:
# Loading Member 1 Outputs

company_mapping = pd.read_csv("../data/processed/company_mapping.csv")

company_mapping.columns = (
    company_mapping.columns
    .str.strip()
    .str.lower()
)

print("Company Mapping Loaded Successfully")

display(company_mapping.head())

Company Mapping Loaded Successfully


,ticker,company_name,cik,sector,year
0,APTV,Aptiv PLC,1521332,Consumer Cyclical,2014
1,APTV,Aptiv PLC,1521332,Consumer Cyclical,2015
2,APTV,Aptiv PLC,1521332,Consumer Cyclical,2016
3,APTV,Aptiv PLC,1521332,Consumer Cyclical,2017
4,APTV,Aptiv PLC,1521332,Consumer Cyclical,2018


#### Step 3 - Validating datasets

In [3]:
# Validate Input Dataset

print("="*60)
print("DATASET SUMMARY")
print("="*60)

print("Companies :", company_mapping["ticker"].nunique())
print("Years :", sorted(company_mapping["year"].unique()))
print("Rows :", len(company_mapping))

DATASET SUMMARY
Companies : 22
Years : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Rows : 110


#### Step 4 - Locating EDGAR dataset

In [4]:
# Locating EDGAR Dataset

EDGAR_PATH = Path("../data/edgar")

print(EDGAR_PATH.resolve())

C:\Rakhi's Workplace\NCI\MODULES - SEM 2\2. Deep Learning and Generative AI\AI_Financial_Analyst\data\edgar


#### Step 5 - Indexing all JSON files

In [5]:
# Index EDGAR Files

json_files = sorted(EDGAR_PATH.rglob("*.json"))

print("="*60)
print("TOTAL JSON FILES")
print("="*60)

print(len(json_files))

TOTAL JSON FILES
36557


#### Step 6 - Building filing metadata

In [6]:
# Building Filing Metadata

records = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:

        filing = json.load(f)

    records.append({

        "file_path": str(file),
        "cik": filing["cik"],
        "year": filing["year"]

    })

filings_df = pd.DataFrame(records)

print(filings_df.dtypes)

display(filings_df.head())

file_path    object
cik          object
year         object
dtype: object


,file_path,cik,year
0,..\data\edgar\2014\1000045_2014.json,1000045,2014
1,..\data\edgar\2014\1000180_2014.json,1000180,2014
2,..\data\edgar\2014\1000209_2014.json,1000209,2014
3,..\data\edgar\2014\1000228_2014.json,1000228,2014
4,..\data\edgar\2014\1000229_2014.json,1000229,2014


#### Step 7 - Standardising Data Types

Before matching the structured financial dataset with the EDGAR dataset,
the CIK and reporting year fields are standardised.

This ensures that both datasets use the same data types and prevents
matching errors during the merge operation.

In [7]:
# Standardise Data Types

# Company Mapping
company_mapping["cik"] = company_mapping["cik"].astype(str).str.strip()
company_mapping["year"] = company_mapping["year"].astype(str)

# EDGAR Metadata
filings_df["cik"] = filings_df["cik"].astype(str).str.strip()
filings_df["year"] = filings_df["year"].astype(str)

print("Data types standardised successfully.")

Data types standardised successfully.


#### Step 8 - Matching Structured and Narrative Data

The structured financial dataset prepared by Member 1 is matched with the
historical EDGAR dataset using two common identifiers:

- Central Index Key (CIK)
- Reporting Year

This ensures that the financial indicators and narrative sections refer to
the same company and reporting period.

In [8]:
# Matching Company-Year Records

matched_df = company_mapping.merge(

    filings_df,

    on=["cik", "year"],

    how="inner"

)

print("=" * 60)
print("MATCHING RESULTS")
print("=" * 60)

print("Company Mapping Records :", len(company_mapping))
print("Matched Records         :", len(matched_df))

display(matched_df.head())

MATCHING RESULTS
Company Mapping Records : 110
Matched Records         : 109


,ticker,company_name,cik,sector,year,file_path
0,APTV,Aptiv PLC,1521332,Consumer Cyclical,2014,..\data\edgar\2014\1521332_2014.json
1,APTV,Aptiv PLC,1521332,Consumer Cyclical,2015,..\data\edgar\2015\1521332_2015.json
2,APTV,Aptiv PLC,1521332,Consumer Cyclical,2016,..\data\edgar\2016\1521332_2016.json
3,APTV,Aptiv PLC,1521332,Consumer Cyclical,2018,..\data\edgar\2018\1521332_2018.json
4,ARTNA,ARTESIAN RESOURCES CORP,863110,Utilities,2014,..\data\edgar\2014\863110_2014.json


#### Step 8 - Matching company-year records

In [9]:
# Identifying Missing Matches

missing = company_mapping.merge(
    filings_df,
    on=["cik", "year"],
    how="left",
    indicator=True

)

missing = missing[missing["_merge"] == "left_only"]

print("=" * 60)
print("MISSING RECORDS")
print("=" * 60)

print("Number of Missing Records:", len(missing))

if len(missing) > 0:
    display(
        missing[
            [
                "ticker",
                "company_name",
                "cik",
                "year"
            ]
        ]
    )
else:
    print("All company-year combinations matched successfully.")

MISSING RECORDS
Number of Missing Records: 1


,ticker,company_name,cik,year
3,APTV,Aptiv PLC,1521332,2017


#### Step 9 - Investigating the single missing APTV (2017) record

In [10]:
# Investigating Missing Record

aptv_mapping = company_mapping[
    company_mapping["ticker"] == "APTV"
]

display(aptv_mapping)

aptv_edgar = filings_df[
    filings_df["cik"] == "1521332"
]

display(aptv_edgar)

,ticker,company_name,cik,sector,year
0,APTV,Aptiv PLC,1521332,Consumer Cyclical,2014
1,APTV,Aptiv PLC,1521332,Consumer Cyclical,2015
2,APTV,Aptiv PLC,1521332,Consumer Cyclical,2016
3,APTV,Aptiv PLC,1521332,Consumer Cyclical,2017
4,APTV,Aptiv PLC,1521332,Consumer Cyclical,2018


,file_path,cik,year
4076,..\data\edgar\2014\1521332_2014.json,1521332,2014
11467,..\data\edgar\2015\1521332_2015.json,1521332,2015
18616,..\data\edgar\2016\1521332_2016.json,1521332,2016
32428,..\data\edgar\2018\1521332_2018.json,1521332,2018


The investigation confirmed that the historical EDGAR dataset does not
contain a filing for APTV in 2017. This record was therefore excluded from
the narrative dataset.

#### Step 10 - Creating the Matched Dataset

The company mapping and EDGAR metadata are merged using the company CIK
and reporting year.

One company-year combination (APTV, 2017) was not available in the
historical EDGAR dataset and was excluded from further narrative analysis.

The remaining matched records are used to extract the required narrative
sections.

In [11]:
# Keeping Successfully Matched Records

matched_df = matched_df.copy()

print("=" * 60)
print("FINAL MATCHED DATASET")
print("=" * 60)

print(f"Total Matched Records : {len(matched_df)}")
print(f"Unique Companies      : {matched_df['ticker'].nunique()}")

display(matched_df.head())

FINAL MATCHED DATASET
Total Matched Records : 109
Unique Companies      : 22


,ticker,company_name,cik,sector,year,file_path
0,APTV,Aptiv PLC,1521332,Consumer Cyclical,2014,..\data\edgar\2014\1521332_2014.json
1,APTV,Aptiv PLC,1521332,Consumer Cyclical,2015,..\data\edgar\2015\1521332_2015.json
2,APTV,Aptiv PLC,1521332,Consumer Cyclical,2016,..\data\edgar\2016\1521332_2016.json
3,APTV,Aptiv PLC,1521332,Consumer Cyclical,2018,..\data\edgar\2018\1521332_2018.json
4,ARTNA,ARTESIAN RESOURCES CORP,863110,Utilities,2014,..\data\edgar\2014\863110_2014.json


#### Step 11 - Extracting Narrative Sections

For each matched company-year record, the corresponding EDGAR JSON file is
loaded and the following narrative sections are extracted:

- Section 1 - Business
- Section 1A - Risk Factors
- Section 7 - Management Discussion and Analysis (MD&A)

The extracted sections are combined into a single structured dataset for
subsequent text preprocessing and embedding generation.

The historical EDGAR dataset contains multiple sections from the annual
10-K filings. Only Section 1, Section 1A and Section 7 are extracted because they
provide the business overview, principal risks and management discussion
required for the three analytical tasks defined in this project.

In [12]:
# Extracting Narrative Sections

narrative_records = []

for _, row in matched_df.iterrows():
    with open(row["file_path"], "r", encoding="utf-8") as f:
        filing = json.load(f)

    narrative_records.append({

        "ticker": row["ticker"],
        "company_name": row["company_name"],
        "cik": row["cik"],
        "year": row["year"],

        "section_1": filing.get("section_1", ""),
        "section_1A": filing.get("section_1A", ""),
        "section_7": filing.get("section_7", "")

    })

narrative_df = pd.DataFrame(narrative_records)
print("Narrative extraction completed.")

Narrative extraction completed.


#### Step 12 - Validating extracted narratives

In [13]:
# Validating Narrative Dataset

print("=" * 60)
print("NARRATIVE DATASET")
print("=" * 60)

print("Rows :", len(narrative_df))
print("Columns :", len(narrative_df.columns))

display(narrative_df.head())

NARRATIVE DATASET
Rows : 109
Columns : 7


,ticker,company_name,cik,year,section_1,section_1A,section_7
0,APTV,Aptiv PLC,1521332,2014,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
1,APTV,Aptiv PLC,1521332,2015,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
2,APTV,Aptiv PLC,1521332,2016,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
3,APTV,Aptiv PLC,1521332,2018,"ITEM 1. BUSINESS\n“Aptiv,” the “Company,” “we,...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
4,ARTNA,ARTESIAN RESOURCES CORP,863110,2014,ITEM 1. BUSINESS\nGeneral Information\nArtesia...,ITEM 1A. RISK FACTORS\nWe are exposed to a var...,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...


#### Step 13 - Data quality checks

In [14]:
# Data Quality Checks - Missing Narrative Text

required_sections = ["section_1", "section_1A", "section_7"]

for section in required_sections:
    empty = (
        narrative_df[section]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{section}: {empty} empty records")

section_1: 2 empty records
section_1A: 5 empty records
section_7: 0 empty records


#### Step 13A - Narrative Text Length Statistics

After confirming that the required narrative sections are present, the
average text length of each section is calculated.

This provides an indication of the amount of narrative information available
for subsequent text preprocessing and embedding generation.

In [15]:
# Narrative Text Length Statistics

print("=" * 60)
print("TEXT LENGTH STATISTICS")
print("=" * 60)

for section in required_sections:

    lengths = narrative_df[section].fillna("").str.len()

    print(f"\n{section}")
    print(f"Average Characters : {round(lengths.mean()):,}")
    print(f"Minimum Characters : {lengths.min():,}")
    print(f"Maximum Characters : {lengths.max():,}")

TEXT LENGTH STATISTICS

section_1
Average Characters : 33,980
Minimum Characters : 0
Maximum Characters : 94,356

section_1A
Average Characters : 54,740
Minimum Characters : 0
Maximum Characters : 136,283

section_7
Average Characters : 60,150
Minimum Characters : 272
Maximum Characters : 153,698


#### Step 14 - Extraction summary

In [16]:
# Extraction Summary

print("=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    "Metric": [
        "Company-Year Records",
        "Successfully Matched",
        "Missing Records",
        "Narrative Records Extracted"
    ],
    "Value": [
        len(company_mapping),
        len(matched_df),
        len(missing),
        len(narrative_df)
    ]
})

display(summary)

EXTRACTION SUMMARY


,Metric,Value
0,Company-Year Records,110
1,Successfully Matched,109
2,Missing Records,1
3,Narrative Records Extracted,109


#### Step 14A - Final Dataset Schema

The structure of the extracted narrative dataset is displayed to verify
that all required variables are available before saving the dataset.

In [17]:
# Final Dataset Schema

print("="*60)
print("FINAL DATASET SCHEMA")
print("="*60)

print(narrative_df.dtypes)

display(narrative_df.head())

FINAL DATASET SCHEMA
ticker          object
company_name    object
cik             object
year            object
section_1       object
section_1A      object
section_7       object
dtype: object


,ticker,company_name,cik,year,section_1,section_1A,section_7
0,APTV,Aptiv PLC,1521332,2014,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
1,APTV,Aptiv PLC,1521332,2015,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
2,APTV,Aptiv PLC,1521332,2016,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
3,APTV,Aptiv PLC,1521332,2018,"ITEM 1. BUSINESS\n“Aptiv,” the “Company,” “we,...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
4,ARTNA,ARTESIAN RESOURCES CORP,863110,2014,ITEM 1. BUSINESS\nGeneral Information\nArtesia...,ITEM 1A. RISK FACTORS\nWe are exposed to a var...,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...


#### Step 15 - Saving as CSV and Parquet

In [18]:
# Saving Narrative Dataset (CSV + Parquet)

print("Final Dataset Columns")
print(narrative_df.columns.tolist())

OUTPUT_PATH_CSV = Path("../data/processed/narrative_sections.csv")
OUTPUT_PATH_PARQUET = Path("../data/processed/narrative_sections.parquet")

# CSV - for inspection
narrative_df.to_csv(OUTPUT_PATH_CSV, index=False)

# Parquet - for Notebook 4 onwards
narrative_df.to_parquet(OUTPUT_PATH_PARQUET, index=False)

Final Dataset Columns
['ticker', 'company_name', 'cik', 'year', 'section_1', 'section_1A', 'section_7']


In [19]:
print("="*60)
print("FILES SAVED")
print("="*60)

print("Narrative dataset saved successfully.")
print(f"CSV File      : {OUTPUT_PATH_CSV.resolve()}")
print(f"Parquet File : {OUTPUT_PATH_PARQUET.resolve()}")

FILES SAVED
Narrative dataset saved successfully.
CSV File      : C:\Rakhi's Workplace\NCI\MODULES - SEM 2\2. Deep Learning and Generative AI\AI_Financial_Analyst\data\processed\narrative_sections.csv
Parquet File : C:\Rakhi's Workplace\NCI\MODULES - SEM 2\2. Deep Learning and Generative AI\AI_Financial_Analyst\data\processed\narrative_sections.parquet


#### Conclusion

This notebook successfully integrated the structured financial dataset
prepared by Member 1 with the historical EDGAR narrative dataset.

Using the company CIK and reporting year, 109 company-year records were
matched successfully.

The required narrative sections (Business, Risk Factors and Management
Discussion & Analysis) were extracted and saved as both CSV and Parquet
files.

One company-year combination (APTV, 2017) was unavailable in the historical
EDGAR dataset and was excluded from subsequent analysis.

The extracted narrative dataset provides the foundation for Notebook 4,
where the text will be cleaned, chunked and prepared for embedding
generation.